# Milestone 1 - NLP Basics & Similarity Metrics

Working on data cleaning, basic vocabulary stats, and TF-IDF cosine similarity.

In [ ]:
import string
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

# Load data
df = pd.read_csv('../guides/train.csv')
print(df.shape)
df.head()

### Q1: Option Frequency Distribution
Checking how many times each option (A, B, C, D, E) appears as correct answer.

In [ ]:
counts = df['answer'].value_counts()
print(counts)

# sum of most frequent + least frequent
ans_q1 = counts.max() + counts.min()
print("Q1 sum:", ans_q1)

### Q2 & Q3: Text Cleaning and Stop Words
Lowercase + strip punctuation to get vocab size, then filter stop words for row ID 1.

In [ ]:
def clean_text(text):
    text = str(text).lower()
    return text.translate(str.maketrans('', '', string.punctuation))

cleaned_prompts = df['prompt'].apply(clean_text)

# Q2: Total unique words
vocab = set()
for p in cleaned_prompts:
    vocab.update(p.split())
print("Q2 Vocab size:", len(vocab))

# Q3: Stop words left in Row ID 1 prompt
row1_clean = cleaned_prompts.iloc[1]
words_left = [w for w in row1_clean.split() if w not in ENGLISH_STOP_WORDS]
print("Q3 Words remaining in row 1:", len(words_left))

### Q4, Q5 & Q6: TF-IDF & Cosine Similarity
Fitting TF-IDF vectorizer on combined text, testing similarity on Row ID 1, and checking overall dataset accuracy.

In [ ]:
# Q4: Fit TfidfVectorizer on combined prompts + options
combined = []
for _, row in df.iterrows():
    combined.append(f"{row['prompt']} {row['A']} {row['B']} {row['C']} {row['D']} {row['E']}")

vec = TfidfVectorizer(stop_words='english')
vec.fit(combined)
print("Q4 Feature vocab size:", len(vec.get_feature_names_out()))

# Q5: Cosine sim between Row ID 1 prompt and Option A
p_vec = vec.transform([df.iloc[1]['prompt']])
a_vec = vec.transform([df.iloc[1]['A']])
sim_a = cosine_similarity(p_vec, a_vec)[0][0]
print("Q5 Row 1 prompt vs option A cos sim:", round(sim_a, 4))

# Q6: Checking % where highest cosine similarity matches correct answer
opts = ['A', 'B', 'C', 'D', 'E']
correct_cnt = 0

for _, row in df.iterrows():
    pv = vec.transform([row['prompt']])
    sims = [cosine_similarity(pv, vec.transform([row[o]]))[0][0] for o in opts]
    best_opt = opts[np.argmax(sims)]
    if best_opt == row['answer']:
        correct_cnt += 1

acc_pct = (correct_cnt / len(df)) * 100
print("Q6 Accuracy %:", round(acc_pct, 2))